# Notebook 6: Phase-C Selective Correction from z

This notebook focuses only on `phase_c.pt` and asks a practical question: when the frozen backbone is uncertain on hard examples, can `z` provide a useful selective correction signal?


In [ ]:
# Colab-first setup: clone/update the repo, install it, and mount Drive.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/jacobposchl/model-interpretability.git"
REPO_DIR = Path("/content/model_interpretability")

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    print("Running outside Colab; using the current working tree.")


In [ ]:
from flow_circuits.data import CIFAR10_STATS, build_cifar10_splits
from flow_circuits.evaluation import (
    NB06_EXPERIMENT_IDS,
    run_confidence_and_calibration_experiment,
    run_hard_example_audit_experiment,
    run_hard_pair_case_study_experiment,
    run_selective_hybrid_correction_experiment,
)
from flow_circuits.training import collect_interpretability_outputs, load_components_from_checkpoint


## Config

Change the single `EXPERIMENTS` line below to run all four `nb06` experiments or any subset.


In [ ]:
from pathlib import Path

RUN_MODE = "full"  # "debug" or "full"
CONFIG_NAME = "resnet18_aligned"
EXPERIMENTS = "all"
FORCE_RERUN = False

TRIGGER_MODE = "hard_pair_top2_and_low_margin"
MARGIN_THRESHOLD = None
MARGIN_QUANTILE = 0.25
ENTROPY_THRESHOLD = None
ENTROPY_QUANTILE = 0.75

DRIVE_ROOT = Path("/content/drive/MyDrive/flow_circuits")
EXPERIMENT_ROOT = DRIVE_ROOT / "experiments" / CONFIG_NAME
OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb06_hard_pair_correction_from_z" / CONFIG_NAME

PHASE_C_CHECKPOINT = EXPERIMENT_ROOT / "phase_c.pt"


In [ ]:
import json
import os
import time

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch
from IPython.display import display

try:
    import pandas as pd
except ImportError:
    pd = None

CHECKPOINT_TAG = "phase_c"
CHECKPOINT_LABEL = "Phase C"
EXPERIMENT_LABELS = {
    "hard_example_audit": "Hard Example Audit",
    "selective_hybrid_correction": "Selective Hybrid Correction",
    "confidence_and_calibration": "Confidence And Calibration",
    "correction_case_studies": "Correction Case Studies",
}
AUTO_N_JOBS = max(1, min(8, (os.cpu_count() or 1) - 1 if (os.cpu_count() or 1) > 1 else 1))

if EXPERIMENTS == "all":
    SELECTED_EXPERIMENTS = list(NB06_EXPERIMENT_IDS)
else:
    SELECTED_EXPERIMENTS = [str(name) for name in EXPERIMENTS]
invalid = sorted(set(SELECTED_EXPERIMENTS) - set(NB06_EXPERIMENT_IDS))
if invalid:
    raise ValueError(f"Unknown experiments: {invalid}. Valid ids: {NB06_EXPERIMENT_IDS}")

if not PHASE_C_CHECKPOINT.exists():
    raise FileNotFoundError(f"Missing required checkpoint: {PHASE_C_CHECKPOINT}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = OUTPUT_DIR / CHECKPOINT_TAG
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SETTINGS_BY_MODE = {
    "debug": {
        "hard_example_audit": {"fit_max_images": 1000, "val_max_images": 500, "test_max_images": 500, "top_pairs": 3, "top_k_nodes": 3, "n_jobs": AUTO_N_JOBS},
        "selective_hybrid_correction": {"fit_max_images": 1000, "val_max_images": 500, "test_max_images": 500, "top_pairs": 3, "top_k_nodes": 3, "n_jobs": AUTO_N_JOBS},
        "confidence_and_calibration": {"fit_max_images": 1000, "val_max_images": 500, "test_max_images": 500, "top_pairs": 3, "top_k_nodes": 3, "n_jobs": AUTO_N_JOBS},
        "correction_case_studies": {"fit_max_images": 1000, "val_max_images": 500, "test_max_images": 500, "top_pairs": 3, "top_k_nodes": 3, "exemplar_count": 6, "n_jobs": AUTO_N_JOBS},
    },
    "full": {
        "hard_example_audit": {"fit_max_images": 4000, "val_max_images": 1000, "test_max_images": 1000, "top_pairs": 5, "top_k_nodes": 5, "n_jobs": AUTO_N_JOBS},
        "selective_hybrid_correction": {"fit_max_images": 4000, "val_max_images": 1000, "test_max_images": 1000, "top_pairs": 5, "top_k_nodes": 5, "n_jobs": AUTO_N_JOBS},
        "confidence_and_calibration": {"fit_max_images": 4000, "val_max_images": 1000, "test_max_images": 1000, "top_pairs": 5, "top_k_nodes": 5, "n_jobs": AUTO_N_JOBS},
        "correction_case_studies": {"fit_max_images": 4000, "val_max_images": 1000, "test_max_images": 1000, "top_pairs": 5, "top_k_nodes": 5, "exemplar_count": 6, "n_jobs": AUTO_N_JOBS},
    },
}
if RUN_MODE not in SETTINGS_BY_MODE:
    raise ValueError(f"Unsupported RUN_MODE: {RUN_MODE}")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
components = load_components_from_checkpoint(PHASE_C_CHECKPOINT, device=device)
base_config = components.config
loaders = build_cifar10_splits(
    data_dir=base_config["data"]["data_dir"],
    batch_size=base_config["data"]["batch_size"],
    num_workers=base_config["data"].get("num_workers", 4),
    seed=base_config["data"].get("seed", 0),
    augment_fit=base_config["data"].get("augment_fit", True),
    download=base_config["data"].get("download", True),
)

RESULTS = {}
_OUTPUT_CACHE = {}


def _should_run(experiment_id):
    return experiment_id in SELECTED_EXPERIMENTS


def _cache_path(experiment_id):
    return RESULTS_DIR / f"{experiment_id}.json"


def _format_seconds(seconds):
    seconds = max(int(seconds), 0)
    minutes, seconds = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        return f"{hours}h {minutes}m {seconds}s"
    if minutes:
        return f"{minutes}m {seconds}s"
    return f"{seconds}s"


def _progress_logger(**event):
    stamp = time.strftime("%H:%M:%S")
    experiment = event.get("experiment")
    stage = str(event.get("stage", "")).replace("_", " ")
    completed = event.get("completed")
    total = event.get("total")
    elapsed = _format_seconds(event.get("elapsed_seconds", 0.0))
    eta = event.get("eta_seconds")
    message = event.get("message", "")
    if total:
        prefix = f"{stage} {completed}/{total}"
        eta_text = f" | ETA {_format_seconds(eta)}" if eta is not None else ""
    else:
        prefix = stage
        eta_text = ""
    print(f"[{stamp}] {CHECKPOINT_LABEL} | {experiment} | {prefix} | elapsed {elapsed}{eta_text} | {message}")


def _run_or_load(experiment_id, runner):
    cache_path = _cache_path(experiment_id)
    if cache_path.exists() and not FORCE_RERUN:
        print(f"Loading cached {experiment_id}: {cache_path}")
        return json.loads(cache_path.read_text(encoding="utf-8"))
    print(f"Running {experiment_id} ...")
    return runner(cache_path)


def _write_summary_table(rows, title):
    print(title)
    if not rows:
        print("(no rows)")
        return
    if pd is None:
        for row in rows:
            print(row)
        return
    display(pd.DataFrame(rows))


def _denormalize_image(image_tensor):
    mean = torch.tensor(CIFAR10_STATS["mean"], dtype=image_tensor.dtype).view(3, 1, 1)
    std = torch.tensor(CIFAR10_STATS["std"], dtype=image_tensor.dtype).view(3, 1, 1)
    image = image_tensor.cpu() * std + mean
    return image.clamp(0.0, 1.0).permute(1, 2, 0).numpy()


def _draw_overlay(ax, overlay):
    if not overlay:
        return
    for box in overlay.get("active_boxes", []):
        color = "red" if box.get("is_representative") else "cyan"
        rect = patches.Rectangle((box["x0"], box["y0"]), box["x1"] - box["x0"], box["y1"] - box["y0"], linewidth=2, edgecolor=color, facecolor="none")
        ax.add_patch(rect)


def _get_outputs(split_name, max_images):
    key = (split_name, int(max_images))
    if key not in _OUTPUT_CACHE:
        _OUTPUT_CACHE[key] = collect_interpretability_outputs(
            components,
            loaders[split_name],
            device=device,
            max_images=max_images,
        )
    return _OUTPUT_CACHE[key]


def _show_examples(example_rows, outputs, title, subtitle_key="true_label_name"):
    if not example_rows:
        print(f"{title}: no examples")
        return
    ncols = min(3, len(example_rows))
    nrows = int(np.ceil(len(example_rows) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.atleast_1d(axes).reshape(nrows, ncols)
    for ax in axes.flatten():
        ax.axis("off")
    for ax, row in zip(axes.flatten(), example_rows):
        image = _denormalize_image(outputs["images"][row["row_index"]])
        ax.imshow(image)
        _draw_overlay(ax, row.get("overlay"))
        label = row.get(subtitle_key, row.get("true_label_name", "?"))
        ax.set_title(f"idx={row['dataset_index']} | label={label}")
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def _show_node_heatmap(rows, value_key, title):
    if not rows:
        print(f"{title}: no rows")
        return
    n_layers = 1 + max(int(row["layer_idx"]) for row in rows)
    n_cells = 1 + max(int(row["cell_idx"]) for row in rows)
    grid = np.full((n_layers, n_cells), np.nan, dtype=float)
    for row in rows:
        grid[int(row["layer_idx"]), int(row["cell_idx"])] = float(row[value_key])
    fig, ax = plt.subplots(figsize=(max(6, n_cells / 2), max(3, n_layers / 1.5)))
    im = ax.imshow(grid, aspect="auto", cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("cell_idx")
    ax.set_ylabel("layer_idx")
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


## Experiment 1: Hard Example Audit

Confirm that `Phase C z` contains broad class information and inspect the hardest validation-selected confusion pairs.


In [ ]:
if _should_run("hard_example_audit"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["hard_example_audit"]
    RESULTS["hard_example_audit"] = _run_or_load(
        "hard_example_audit",
        lambda cache_path: run_hard_example_audit_experiment(
            components,
            loaders["fit"],
            loaders["val"],
            loaders["test"],
            device=device,
            checkpoint_tag=CHECKPOINT_TAG,
            fit_max_images=settings["fit_max_images"],
            val_max_images=settings["val_max_images"],
            test_max_images=settings["test_max_images"],
            top_pairs=settings["top_pairs"],
            top_k_nodes=settings["top_k_nodes"],
            n_jobs=settings["n_jobs"],
            output_path=cache_path,
            progress_callback=_progress_logger,
        ),
    )
else:
    print("Skipping hard_example_audit.")


In [ ]:
if "hard_example_audit" in RESULTS:
    audit = RESULTS["hard_example_audit"]
    _write_summary_table([audit["summary"]], title="Hard-example audit summary")
    multiclass = audit["multiclass_probe_audit"]
    hard_pairs = audit["hard_pair_probe_benchmark"]
    _write_summary_table([multiclass["summary"]], title="Multiclass probe audit summary")
    _write_summary_table(multiclass["top_nodes_by_accuracy"], title="Top nodes by held-out multiclass accuracy")
    _show_node_heatmap(multiclass["per_node"], "accuracy", "Per-node multiclass accuracy heatmap")
    _write_summary_table([hard_pairs["summary"]], title="Hard-pair benchmark summary")
    _write_summary_table(hard_pairs["pair_rows"], title="Hard-pair benchmark rows")
    if hard_pairs["pair_rows"]:
        _write_summary_table(hard_pairs["pair_rows"][0]["selected_top_nodes"], title="Top nodes for hardest pair")


## Experiment 2: Selective Hybrid Correction

Trigger correction only on Phase-C hard examples, then compare backbone, full-`z`, and top-node hybrid performance.


In [ ]:
if _should_run("selective_hybrid_correction"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["selective_hybrid_correction"]
    RESULTS["selective_hybrid_correction"] = _run_or_load(
        "selective_hybrid_correction",
        lambda cache_path: run_selective_hybrid_correction_experiment(
            components,
            loaders["fit"],
            loaders["val"],
            loaders["test"],
            device=device,
            checkpoint_tag=CHECKPOINT_TAG,
            fit_max_images=settings["fit_max_images"],
            val_max_images=settings["val_max_images"],
            test_max_images=settings["test_max_images"],
            top_pairs=settings["top_pairs"],
            top_k_nodes=settings["top_k_nodes"],
            trigger_mode=TRIGGER_MODE,
            margin_threshold=MARGIN_THRESHOLD,
            margin_quantile=MARGIN_QUANTILE,
            entropy_threshold=ENTROPY_THRESHOLD,
            entropy_quantile=ENTROPY_QUANTILE,
            n_jobs=settings["n_jobs"],
            output_path=cache_path,
            progress_callback=_progress_logger,
        ),
    )
else:
    print("Skipping selective_hybrid_correction.")


In [ ]:
if "selective_hybrid_correction" in RESULTS:
    result = RESULTS["selective_hybrid_correction"]
    _write_summary_table([result["summary"]], title="Selective hybrid correction summary")
    _write_summary_table(result["backbone"]["per_pair_rows"], title="Backbone trigger-subset per-pair rows")
    _write_summary_table(result["full_z_hybrid"]["per_pair_rows"], title="Full-z hybrid per-pair rows")
    _write_summary_table(result["top_node_hybrid"]["per_pair_rows"], title="Top-node hybrid per-pair rows")


## Experiment 3: Confidence And Calibration

Measure whether the selective Phase-C correction signal helps confidence quality and error detection, not just top-1 accuracy.


In [ ]:
if _should_run("confidence_and_calibration"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["confidence_and_calibration"]
    RESULTS["confidence_and_calibration"] = _run_or_load(
        "confidence_and_calibration",
        lambda cache_path: run_confidence_and_calibration_experiment(
            components,
            loaders["fit"],
            loaders["val"],
            loaders["test"],
            device=device,
            checkpoint_tag=CHECKPOINT_TAG,
            fit_max_images=settings["fit_max_images"],
            val_max_images=settings["val_max_images"],
            test_max_images=settings["test_max_images"],
            top_pairs=settings["top_pairs"],
            top_k_nodes=settings["top_k_nodes"],
            trigger_mode=TRIGGER_MODE,
            margin_threshold=MARGIN_THRESHOLD,
            margin_quantile=MARGIN_QUANTILE,
            entropy_threshold=ENTROPY_THRESHOLD,
            entropy_quantile=ENTROPY_QUANTILE,
            n_jobs=settings["n_jobs"],
            output_path=cache_path,
            progress_callback=_progress_logger,
        ),
    )
else:
    print("Skipping confidence_and_calibration.")


In [ ]:
if "confidence_and_calibration" in RESULTS:
    result = RESULTS["confidence_and_calibration"]
    _write_summary_table([result["summary"]], title="Confidence and calibration summary")
    _write_summary_table([result["overall"]["backbone"]], title="Backbone confidence summary")
    _write_summary_table([result["overall"]["full_z_hybrid"]], title="Full-z hybrid confidence summary")
    _write_summary_table([result["overall"]["top_node_hybrid"]], title="Top-node hybrid confidence summary")
    _write_summary_table([result["triggered_subset"]["backbone_error_detection"]], title="Triggered-subset backbone error detection")
    _write_summary_table([result["triggered_subset"]["full_z_probe_error_detection"]], title="Triggered-subset full-z error detection")
    _write_summary_table([result["triggered_subset"]["top_node_probe_error_detection"]], title="Triggered-subset top-node error detection")


## Experiment 4: Correction Case Studies

Render corrected, harmed, and unchanged Phase-C examples so the selective correction behavior is inspectable in image space.


In [ ]:
if _should_run("correction_case_studies"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["correction_case_studies"]
    RESULTS["correction_case_studies"] = _run_or_load(
        "correction_case_studies",
        lambda cache_path: run_hard_pair_case_study_experiment(
            components,
            loaders["fit"],
            loaders["val"],
            loaders["test"],
            device=device,
            checkpoint_tag=CHECKPOINT_TAG,
            fit_max_images=settings["fit_max_images"],
            val_max_images=settings["val_max_images"],
            test_max_images=settings["test_max_images"],
            top_pairs=settings["top_pairs"],
            top_k_nodes=settings["top_k_nodes"],
            exemplar_count=settings["exemplar_count"],
            trigger_mode=TRIGGER_MODE,
            margin_threshold=MARGIN_THRESHOLD,
            margin_quantile=MARGIN_QUANTILE,
            entropy_threshold=ENTROPY_THRESHOLD,
            entropy_quantile=ENTROPY_QUANTILE,
            n_jobs=settings["n_jobs"],
            output_path=cache_path,
            progress_callback=_progress_logger,
        ),
    )
else:
    print("Skipping correction_case_studies.")


In [ ]:
if "correction_case_studies" in RESULTS:
    result = RESULTS["correction_case_studies"]
    settings = SETTINGS_BY_MODE[RUN_MODE]["correction_case_studies"]
    outputs = _get_outputs("test", settings["test_max_images"])
    _write_summary_table([result["summary"]], title="Correction case study summary")
    for pair_row in result["pair_case_studies"]:
        print()
        print(f"{CHECKPOINT_LABEL} corrected examples: {pair_row['trigger_pair_name']}")
        _show_examples(pair_row["corrected_examples"], outputs, f"{CHECKPOINT_LABEL} corrected: {pair_row['trigger_pair_name']}")
        print(f"{CHECKPOINT_LABEL} harmed examples: {pair_row['trigger_pair_name']}")
        _show_examples(pair_row["harmed_examples"], outputs, f"{CHECKPOINT_LABEL} harmed: {pair_row['trigger_pair_name']}")
        print(f"{CHECKPOINT_LABEL} unchanged examples: {pair_row['trigger_pair_name']}")
        _show_examples(pair_row["unchanged_examples"], outputs, f"{CHECKPOINT_LABEL} unchanged: {pair_row['trigger_pair_name']}")


## Summary

A compact Phase-C-only summary for the experiments that ran in this notebook.


In [ ]:
summary_rows = []
if "hard_example_audit" in RESULTS:
    summary_rows.append({
        "experiment": "Hard Example Audit",
        "value": RESULTS["hard_example_audit"]["summary"]["mean_full_z_probe_accuracy"],
        "metric": "mean_full_z_probe_accuracy",
    })
if "selective_hybrid_correction" in RESULTS:
    summary_rows.append({
        "experiment": "Selective Hybrid Correction",
        "value": RESULTS["selective_hybrid_correction"]["summary"]["top_node_hybrid_overall_accuracy"],
        "metric": "top_node_hybrid_overall_accuracy",
    })
if "confidence_and_calibration" in RESULTS:
    summary_rows.append({
        "experiment": "Confidence And Calibration",
        "value": RESULTS["confidence_and_calibration"]["summary"]["top_node_hybrid_ece"],
        "metric": "top_node_hybrid_ece",
    })
if "correction_case_studies" in RESULTS:
    summary_rows.append({
        "experiment": "Correction Case Studies",
        "value": RESULTS["correction_case_studies"]["summary"]["n_corrected_examples"],
        "metric": "n_corrected_examples",
    })
_write_summary_table(summary_rows, title="Phase-C selective correction summary")
